## PINN Training Notebook

This notebook is the experiment driver.

Data source:
- training data is generated by `traingenerate/generate.py`
- reinitialization snapshots are generated by `traingenerate/reinit.py`

Experiment goal:
- keep the network architecture fixed
- change training hyper-parameters such as learning rate, loss weights, epoch count, and patience
- run training repeatedly and compare curves and validation figures

Core implementation stays in Python modules. The notebook only controls experiments and visualization.


## Part 1: Dataset From `traingenerate`

This notebook does not define the dataset logic.

It uses:
- `DataConfig` from `traingenerate/config.py`
- `generate_training_bundle(...)` from `traingenerate/generate.py`
- `save_training_bundle_hdf5(...)` from `traingenerate/generate.py`

In normal experiments, data-generation parameters are usually kept fixed.
The main trial-and-error loop happens in the training parameters below.


In [9]:
from pathlib import Path

import torch

from model.config import TrainConfig, default_dataset_path
from model.model import ReinitPINN, count_parameters
from model.train import (
    create_optimizer,
    plot_loss_history,
    plot_validation_snapshot,
    plot_weight_schedule,
    summarize_validation_metrics,
    train_model,
)
from traingenerate.config import DataConfig, ReinitConfig
from traingenerate.generate import default_dataset_filename, describe_bundle, generate_training_bundle, save_training_bundle_hdf5

# Data source: defaults come from traingenerate/config.py.

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
reinit_config = ReinitConfig()
dataset_path = Path("dataset") / default_dataset_filename(reinit_config.cfl)

data_config = DataConfig()
train_bundle = generate_training_bundle(data_config=data_config, reinit_config=reinit_config)
save_training_bundle_hdf5(train_bundle, dataset_path)

describe_bundle(train_bundle)
print(f"Dataset file: {dataset_path.resolve()}")
print(f"Device: {device}")


## Part 2: Experiment Parameters

Default experiment values come from `model/config.py` and `traingenerate/config.py`.

If you want config-first experiments, change those files first, then rerun the notebook.

Typical training changes live in `model/config.py`:
- `activation`
- `lr`
- `max_epochs`
- `patience`
- `lambda_ic`
- `lambda_snapshot`
- `lambda_pde`
- `pde_warmup_epochs`

The model architecture stays in `model/model.py`.


In [ ]:
# Training configuration: defaults come from model/config.py.
# Override here only for one-off experiments.

train_config = TrainConfig(dataset_path=dataset_path)

checkpoint_path = Path("best_reinit_pinn.pt")
reinit_model = ReinitPINN(hidden_units=train_config.hidden_units, activation=train_config.activation).to(device)
optimizer = create_optimizer(reinit_model, train_config)

print(reinit_model)
print(f"Parameter count: {count_parameters(reinit_model)}")
print(f"Checkpoint: {checkpoint_path.resolve()}")
print(f"Learning rate: {train_config.lr}")
print(f"Max epochs: {train_config.max_epochs}")


## Part 3: Train And Plot

Run training, then inspect:
- loss curves
- weight schedule and PDE trigger
- validation field plots
- aggregate validation metrics


In [ ]:
reinit_model, history = train_model(
    reinit_model,
    bundle=train_bundle,
    train_config=train_config,
    device=device,
    checkpoint_path=checkpoint_path,
    optimizer=optimizer,
)

print(f"Best checkpoint: {checkpoint_path.resolve()}")
print(f"Best epoch: {history['best_epoch']}")


In [ ]:
plot_loss_history(history)

In [ ]:
plot_weight_schedule(history, train_config)

In [ ]:
validation_metrics = plot_validation_snapshot(
    reinit_model,
    train_bundle,
    data_config=data_config,
    device=device,
)


In [ ]:
all_validation_metrics = summarize_validation_metrics(
    reinit_model,
    train_bundle,
    data_config=data_config,
    device=device,
)
